In [1658]:
from discovery_child_development import PROJECT_DIR
import pandas as pd
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
PATH_TO_DATASET = ENRICHED_DATA_DIR / 'openalex_patents_relevance_labels_only_relevant.csv'

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

38


In [1659]:
# Load the data with texts
text_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
)
len(text_df)

51234

In [1669]:
dfs = []
for topic in topics:
    keywords = topics_dict[topic]["filtering_keywords"]
    df = (
        pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/taxonomy_cat_predictions_{topic}.csv')
        .merge(text_df[['id', 'text']], on='id', how='left')
    )
    keyword_hits = (
        df.text
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
        .str.contains("|".join(keywords))
    )
    df = df[keyword_hits]
    dfs.append(df)

In [1681]:
labelled_df = (
    pd.concat(dfs, ignore_index=True)
    # Key step: taking only data that's robustly relevant
    .query("prediction==1.0")
    .groupby("id")
    .agg(topics = ("topic", list))
    .reset_index()
    # .astype({"topics": str})
)

In [1692]:
text_labelled_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
    .merge(labelled_df, on="id", how="left")
    .assign(topics = lambda df: df.topics.apply(lambda x: ", ".join(x) if type(x) == list else x))
    .drop(columns=["Unnamed: 0", "predictions"])
)

In [1685]:
# Papers with no labels
n_with_topics = (text_labelled_df.topics.isnull() == False).sum()
n_without_topics = text_labelled_df.topics.isnull().sum()
n_with_topics / len(text_labelled_df), n_without_topics / len(text_labelled_df)

(0.8676269664675801, 0.1323730335324199)

In [1693]:
print(n_with_topics)

44452


In [1694]:
text_labelled_df

,id,text,source,topics
0,CN-107945066-A,Kindergarten intelligent control system and co...,patents,robotics
1,WO-2015164890-A2,A method of promoting growth and brain develop...,patents,NaN
2,KR-101597453-B1,Smart sterilizer. The present invention relate...,patents,NaN
3,EP-3787584-A1,Monitoring system for providing both visual an...,patents,"ai2, infancy"
4,US-11844731-B2,Systems and methods for indicating an open por...,patents,"sleep, infancy"
...,...,...,...,...
51229,W4380048305,Trilingual families' language strategies: pote...,openalex,communication
51230,W4385650553,The Effects of Vitamin D Supplementation on Re...,openalex,"rct, nutrition, health"
51231,W4367694030,A Study to Assess the Effectiveness of Video A...,openalex,"social_services, protection"
51232,W4380886404,Vaginal Bleeding In Prepubertal Girls-A Case S...,openalex,NaN


In [1695]:
text_labelled_df.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_filtered.csv', index=False)